In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
patientpayer_table = dbutils.widgets.get("patientpayer_table")
patient_table = dbutils.widgets.get("patient_table")
branch_table = dbutils.widgets.get("branch_table")
payer_table = dbutils.widgets.get("payer_table")
billfrequency_table = dbutils.widgets.get("billfrequency_table")
office_table = dbutils.widgets.get("office_table")
payerbillingcode_table = dbutils.widgets.get("payerbillingcode_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW customfile_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate,
  CAST(FacilityCode AS INT) AS FacilityCode,
  CAST(AcctNbr AS STRING) AS AcctNbr,
  CAST(Company AS STRING) AS Company,
  CAST(Practice AS STRING) AS Practice,
  CAST(BayadaRegion AS STRING) AS BayadaRegion,
  CAST(BayadaDivision AS STRING) AS BayadaDivision,
  CAST(BayadaOfficeAbbreviation AS STRING) AS BayadaOfficeAbbreviation,
  NULL AS CompanyTaxID,
  CAST(ReimbursementTeam AS STRING) AS ReimbursementTeam,
  CAST(PayerSourceNumber AS STRING) AS PayerSourceNumber,
  NULL AS PayerSourceProgramName,
  CAST(BillingPeriodOrFrequency AS STRING) AS BillingPeriodOrFrequency,
  NULL AS EVVRequirements,
  CAST(EVVAggregator AS STRING) AS EVVAggregator,
  NULL AS EVVCodesinScope,
  NULL AS COBPrimaryInsurance, 
  NULL AS ReferralID,
  NULL AS PrimarySubscriberIDNumber,
  CAST(COBStatus AS STRING) AS COBStatus,
  CAST(COBStatus AS STRING) AS COBType,
  CAST(COBEffDate AS STRING) AS COBEffDate,
  CAST(COBLevel AS STRING) AS COBLevel,
  CAST(COBDenialReason AS STRING) AS COBDenialReason,
  CAST(COBSplitDecision AS STRING) AS COBSplitDecision,
  NULL AS LimitedBenefit,
  NULL AS BenefitDate,
  NULL AS BillHoldReason,
  NULL AS BillHoldDate,
  CAST(SourceSystemKey AS INT) AS SourceSystemKey
FROM (
  WITH 
  customfile_cte AS (
    SELECT 
    CAST('{fetch_date}' AS DATE) AS ReportingDate,
    b.ExternalId AS FacilityCode,
    bl.ClaimNumber AS AcctNbr,
    ofc.Company AS Company,
    ofc.Practice AS Practice,
    ofc.Region AS BayadaRegion,
    ofc.Division AS BayadaDivision,
    ofc.OfficeAbbreviation AS BayadaOfficeAbbreviation,
    ofc.ReimbursementOfficeName AS ReimbursementTeam,
    pay.Id AS PayerSourceNumber,
    "Weekly" AS BillingPeriodOrFrequency, --"Weekly" not in billfrequency table.  Always Weekly?
    CASE 
        WHEN pbc.SandataServiceCodeId IS NOT NULL AND pbc.CellTrakServiceCodeId IS NOT NULL THEN 'CellTrak Sandata'
        WHEN pbc.CellTrakServiceCodeId IS NOT NULL THEN 'CellTrak'
        WHEN pbc.SandataServiceCodeId IS NOT NULL THEN 'Sandata'
        ELSE 'None'
    END AS EVVAggregator,
    CASE pp.CobStatus
        WHEN 0 THEN 'Not Selected'
        ELSE CAST(pp.CobStatus AS STRING)
    END AS COBStatus,
    pp.CobType AS COBType,
    -- pp.CobEffectiveStartDate AS COBEffDate, --PDN in example data not matching
    p.ReasonRequested AS COBEffDate,
    CASE pp.CobStatus
        WHEN 0 THEN 'Not Selected'
        ELSE CAST(pp.CobStatus AS STRING)
    END AS COBLevel,
    pp.CobDenialReason AS COBDenialReason,
    pp.SplitDecisionDetails AS COBSplitDecision,
    '19' AS SourceSystemKey
    FROM {source_table} bl
    JOIN {patientpayer_table} pp ON pp.Id = bl.PatientPayerId
    JOIN {patient_table} p ON p.Id = pp.PatientId
    JOIN {branch_table} b ON b.Id = p.BranchId
    JOIN {payer_table} pay ON pay.Id = pp.PayerId
    LEFT JOIN {billfrequency_table} bf ON bf.Id = p.BillFrequencyId
    LEFT JOIN {office_table} ofc ON CAST(b.ExternalId AS INT) = ofc.OfficeNumber
    -- LEFT JOIN {payerbillingcode_table} pbc ON pbc.PayerId = pay.Id
    -- -- EVV Aggregator (pick first payerbillingcode per payer)
    LEFT JOIN (
        SELECT PayerId, SandataServiceCodeId, CellTrakServiceCodeId,
              ROW_NUMBER() OVER (PARTITION BY PayerId ORDER BY Id) as rn
        FROM {payerbillingcode_table}
    ) pbc ON pbc.PayerId = pay.Id AND pbc.rn = 1
    -- WHERE bl.ClaimNumber IN ('317441FJ1301', '307910FI1889', '307954FI1800')
    WHERE bl.isActive='true'
  ),
  customfile_clean AS (
    SELECT *,
    row_number() OVER (PARTITION BY AcctNbr ORDER BY AcctNbr ) AS rn
    FROM customfile_cte
  )
  SELECT 
    ReportingDate, 
    FacilityCode, 
    AcctNbr, 
    Company, 
    Practice, 
    BayadaRegion, 
    BayadaDivision, 
    BayadaOfficeAbbreviation, 
    ReimbursementTeam, 
    PayerSourceNumber,
    BillingPeriodOrFrequency,
    EVVAggregator,
    COBStatus,
    COBType,
    COBEffDate,
    COBLevel,
    COBDenialReason,
    COBSplitDecision,
    SourceSystemKey
  FROM customfile_clean
  WHERE rn=1
)
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING customfile_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 19

WHEN MATCHED THEN
UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.Company = src.Company,
    tgt.Practice = src.Practice,
    tgt.BayadaRegion = src.BayadaRegion,
    tgt.BayadaDivision = src.BayadaDivision,
    tgt.BayadaOfficeAbbreviation = src.BayadaOfficeAbbreviation,
    tgt.CompanyTaxID = src.CompanyTaxID,
    tgt.ReimbursementTeam = src.ReimbursementTeam,
    tgt.PayerSourceNumber = src.PayerSourceNumber,
    tgt.PayerSourceProgramName = src.PayerSourceProgramName,
    tgt.BillingPeriodOrFrequency = src.BillingPeriodOrFrequency,
    tgt.EVVRequirements = src.EVVRequirements,
    tgt.EVVAggregator = src.EVVAggregator,
    tgt.EVVCodesinScope = src.EVVCodesinScope,
    tgt.COBPrimaryInsurance = src.COBPrimaryInsurance,
    tgt.ReferralID = src.ReferralID,
    tgt.PrimarySubscriberIDNumber = src.PrimarySubscriberIDNumber,
    tgt.COBStatus = src.COBStatus,
    tgt.COBType = src.COBType,
    tgt.COBEffDate = src.COBEffDate,
    tgt.COBLevel = src.COBLevel,
    tgt.COBDenialReason = src.COBDenialReason,
    tgt.COBSplitDecision = src.COBSplitDecision,
    tgt.LimitedBenefit = src.LimitedBenefit,
    tgt.BenefitDate = src.BenefitDate,
    tgt.BillHoldReason = src.BillHoldReason,
    tgt.BillHoldDate = src.BillHoldDate,
    tgt.SourceSystemKey = src.SourceSystemKey,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    Company,
    Practice,
    BayadaRegion,
    BayadaDivision,
    BayadaOfficeAbbreviation,
    CompanyTaxID,
    ReimbursementTeam,
    PayerSourceNumber,
    PayerSourceProgramName,
    BillingPeriodOrFrequency,
    EVVRequirements,
    EVVAggregator,
    EVVCodesinScope,
    COBPrimaryInsurance,
    ReferralID,
    PrimarySubscriberIDNumber,
    COBStatus,
    COBType,
    COBEffDate,
    COBLevel,
    COBDenialReason,
    COBSplitDecision,
    LimitedBenefit,
    BenefitDate,
    BillHoldReason,
    BillHoldDate,
    SourceSystemKey,
    _load_timestamp
)
VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.Company,
    src.Practice,
    src.BayadaRegion,
    src.BayadaDivision,
    src.BayadaOfficeAbbreviation,
    src.CompanyTaxID,
    src.ReimbursementTeam,
    src.PayerSourceNumber,
    src.PayerSourceProgramName,
    src.BillingPeriodOrFrequency,
    src.EVVRequirements,
    src.EVVAggregator,
    src.EVVCodesinScope,
    src.COBPrimaryInsurance,
    src.ReferralID,
    src.PrimarySubscriberIDNumber,
    src.COBStatus,
    src.COBType,
    src.COBEffDate,
    src.COBLevel,
    src.COBDenialReason,
    src.COBSplitDecision,
    src.LimitedBenefit,
    src.BenefitDate,
    src.BillHoldReason,
    src.BillHoldDate,
    src.SourceSystemKey,
    current_timestamp()
)
""")
)